In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

## Functions

## Constants 

In [ ]:
ROOT = "/kaggle/input/big-data-derby-2022"
INPUT_DIR = "/kaggle/input/data_preprocessing"

In [ ]:
RACE_PROGRAM_ID = ["race_id", "program_number"]
RACE_TYPES = ["STK", "ALW", "AOC", "MSW"]
RACE_TYPES_FULL = ["Stakes", "Allowance", "Allowance Optional Claiming", "Maiden Special Weight"]
COURSE_TYPES_FULL = ["Hurdle", "Dirt", "Outer turf", "Inner turf", "Turf"]
RACE_TYPE_MAP = {
    "STK": "Stakes",
    "ALW": "Allowance",
    "AOC": "Allowance Optional Claiming",
    "MSW": "Maiden Special Weight"
}
COURSE_TYPE_MAP = {
    "M": "Hurdle",
    "D": "Dirt",
    "O": "Outer turf",
    "I": "Inner turf",
    "T": "Turf"
}

# Race Table
**Contains information regarding the race.**

`track_id` - 3 character id for the track the race took place at. AQU -Aqueduct, BEL - Belmont, SAR - Saratoga.

`race_date` - date the race took place. YYYY-MM-DD.

`race_number` - Number of the race. Passed as 3 characters but can be cast or converted to int for this data set.

`distance_id` - Distance of the race in furlongs passed as an integer. Example - 600 would be 6 furlongs.

`course_type` - The course the race was run over passed as one character. M - Hurdle, D - Dirt, O - Outer turf, I - Inner turf, T - turf.

`track_condition` - The condition of the course the race was run on passed as three characters. YL - Yielding, FM - Firm, SY - Sloppy, GD - Good, FT - Fast, MY - Muddy, SF - Soft.

`run_up_distance` - Distance in feet of the gate to the start of the race passed as an integer.

`race_type` - The classification of the race passed as as five characters. STK - Stakes, WCL - Waiver Claiming, WMC - Waiver Maiden Claiming, SST - Starter Stakes, SHP - Starter Handicap, CLM - Claiming, STR - Starter Allowance, AOC - Allowance Optionl Claimer, SOC - Starter Optional Claimer, MCL - Maiden Claiming, ALW - Allowance, MSW - Maiden Special Weight.

`purse` - Purse in US dollars of the race passed as an money with two decimal places.

`post_time` - Time of day the race began passed as 5 character. Example - 01220 would be 12:20.

In [ ]:
race = pd.read_csv(f"{INPUT_DIR}/race-updated.csv", parse_dates=["race_date"])
race = race[race.race_type.isin(RACE_TYPES)]
race["race_type_full"] = race.race_type.map(RACE_TYPE_MAP)
race["course_type_full"] = race.course_type.map(COURSE_TYPE_MAP)

In [ ]:
start = pd.read_csv(f"{INPUT_DIR}/start-updated.csv", parse_dates=["race_date"])

# Tracking table

**Contains information about the position of each horse throughout the race.**

`track_id` - 3 character id for the track the race took place at. AQU -Aqueduct, BEL - Belmont, SAR - Saratoga.

`race_date` - date the race took place. YYYY-MM-DD.

`race_number` - Number of the race. Passed as 3 characters but can be cast or converted to int for this data set.

`program_number` - Program number of the horse in the race passed as 3 characters. Should remain 3 characters as it isn't limited to just numbers. Is essentially the unique identifier of the horse in the race.

`trakus_index` - The common collection of point of the lat / long of the horse in the race passed as an integer. From what we can tell, it's collected every 0.25 seconds.

`latitude` - The latitude of the horse in the race passed as a float.

`longitude` - The longitude of the horse in the race passed as a float.

In [ ]:
tracking = pd.read_csv(f"{INPUT_DIR}/tracking-updated.csv")
tracking = tracking.sort_values(RACE_PROGRAM_ID)

In [ ]:
distance_cols = [*RACE_PROGRAM_ID, "cumulative_distance"]
race_cols = ["race_id", "track_distance", "course_type_full", "race_type_full"]

In [ ]:
tracking = tracking.merge(race[["race_id", "race_type", "race_type_full", "course_type_full"]], on=["race_id"])

# Distance Analysis

In [ ]:
distance = tracking.loc[tracking.trakus_index.eq(tracking.finish_trakus_index), distance_cols]
distance = distance.merge(race[race_cols], on=["race_id"])
distance = distance.merge(start[[*RACE_PROGRAM_ID, "normalized_place"]], on=RACE_PROGRAM_ID)
distance["distance_vs_track"] = distance["cumulative_distance"].div(distance["track_distance"])

There seems to be a negative association between distance ran and the placing of the horse. The more you run, the better you are likely to place. One hypothesis is that the horses that run more are able to get a better position in the track compared to other horses. Also, when a horse is stronger/fitter, it's also possible that the horse can cover more distance while still be able to run faster than its peers.

In [ ]:
grid = sns.FacetGrid(data=distance, col="course_type_full", col_wrap=3, height=6, aspect=1.5)

grid.map(sns.boxplot, "distance_vs_track")
grid.set_titles(row_template='{row_name}', col_template="Course type: {col_name}")

The distribution of the distances ran in each race type is roughly the same. MSW seems to have more outliers compare to other races.

In [ ]:
grid = sns.FacetGrid(data=distance, col="race_type_full", col_wrap=3, height=6, aspect=1.5)

grid.map(sns.boxplot, "distance_vs_track")
grid.set_titles(col_template="Race type: {col_name}")

There is no correlation between distance and placing after grouping by course type or race type.

In [ ]:
distance.groupby("course_type_full")[["distance_vs_track", "normalized_place"]].corr()

In [ ]:
distance.groupby("race_type_full")[["distance_vs_track", "normalized_place"]].corr()

### Distance during each segment

Overall, there seems to be no impact of distance on the finish position of each horse. However, maybe there's a difference in how the distance ran is allocated throughout the race.

In [ ]:
TRAKUS_INTERVAL = 5 # Intervene every x% of the race.

In [ ]:
tracking["trakus_prop"] = tracking["trakus_index"].div(tracking.groupby("race_id")["trakus_index"].transform("max"))
tracking["trakus_segment"] = tracking["trakus_prop"].floordiv(TRAKUS_INTERVAL / 100).add(1)

In [ ]:
group_cols = [*RACE_PROGRAM_ID, "race_type_full", "course_type_full", "trakus_segment", "finish_trakus_index"]
distance_by_segment = (
    tracking
   .loc[~tracking.dnf & ~tracking.is_finished]
   .groupby(group_cols)["cumulative_distance"]
   .agg(np.ptp)
   .reset_index()
)

In [ ]:
mean_distance_by_segment = distance_by_segment.groupby(["race_id", "trakus_segment"])["cumulative_distance"].transform("mean")
distance_by_segment["relative_distance"] = distance_by_segment["cumulative_distance"].div(mean_distance_by_segment)

distance_by_segment = distance_by_segment.merge(start[[*RACE_PROGRAM_ID, "normalized_place", "position_at_finish"]])

The strategy in all of these races seem to be the same. The distance relative to its peers is roughly the same during the first half of the race while during the second half, the distance increases significantly.

We see in this plot that high placers generally start to run more during the second half of the race compared to its peers. First and second place tend to start running more earlier than third and fourth place. 

Another interesting fact is that in all the race types except Stakes, first place tend to start off running more than its peers, then become more steady, and then run more again during the second half of the race.

In [ ]:
race["race_type"].value_counts()

In [ ]:
POSITIONS_TO_COMPARE = 4
FACET_ROW = "race_type_full"
FACET_COL = "position_at_finish"
POSITION_MASK = distance_by_segment.position_at_finish.le(POSITIONS_TO_COMPARE) 
GROUP_COLS = [FACET_ROW, FACET_COL, "trakus_segment"]

subset_data = distance_by_segment.loc[POSITION_MASK].groupby(GROUP_COLS)["relative_distance"].median().reset_index()

grid = sns.FacetGrid(
    data=subset_data, 
    row=FACET_ROW,
    col=FACET_COL, 
    height=2.5, 
    aspect=1.5,
    margin_titles=True,
    row_order=RACE_TYPES_FULL
)

grid.map(sns.lineplot, "trakus_segment", "relative_distance")
grid.set_titles(row_template='{row_name}', col_template="Position at finish: {col_name}")

for ax in grid.axes.ravel():
    ax.axhline(1, linestyle="--")
    ax.set_ylim([0.98, 1.045])

In [ ]:
subset_data

In [ ]:
race[["race_type_full", "course_type_full"]].value_counts().reset_index().sort_values(["race_type_full", 0], ascending=False)

In [ ]:
POSITIONS_TO_COMPARE = 4
FACET_ROW = "course_type_full"
FACET_COL = "position_at_finish"
POSITION_MASK = distance_by_segment.position_at_finish.le(POSITIONS_TO_COMPARE) 
GROUP_COLS = [FACET_ROW, FACET_COL, "trakus_segment"]

subset_data = distance_by_segment.loc[POSITION_MASK].groupby(GROUP_COLS)["relative_distance"].median().reset_index()

grid = sns.FacetGrid(
    data=subset_data, 
    row=FACET_ROW,
    col=FACET_COL, 
    height=2.5, 
    aspect=1.5,
    margin_titles=True,
    row_order=COURSE_TYPES_FULL
)

grid.map(sns.lineplot, "trakus_segment", "relative_distance")
grid.set_titles(row_template='{row_name}', col_template="Position at finish: {col_name}")

for ax in grid.axes.ravel():
    ax.axhline(1, linestyle="--")
    ax.set_ylim([0.98, 1.05])

# Zone Analysis

## Methodology of defining zone

## Analysis of zone in general

## Analysis of zone-change 

In [ ]:
tracking_sec = tracking.loc[tracking.trakus_index.mod(4).eq(0)].copy().reset_index(drop=True)

In [ ]:
tracking_sec["is_change_zone"] = tracking_sec["zone"].ne(tracking_sec.groupby(RACE_PROGRAM_ID)["zone"].shift(1))

In [ ]:
finish_mask = tracking_sec.trakus_index.le(tracking_sec.finish_trakus_index) & ~tracking_sec.dnf
zone_change = tracking_sec.loc[finish_mask].groupby([*RACE_PROGRAM_ID, "race_type_full", "trakus_segment"])["is_change_zone"].sum().reset_index()

In [ ]:
zone_change[zone_change.is_change_zone.gt(1)]

In [ ]:
zone_change[zone_change.race_id.eq("2019-01-01_1") & zone_change.program_number.eq("3") & zone_change.trakus_segment.eq(2)]

In [ ]:
zone_change = zone_change.merge(start[[*RACE_PROGRAM_ID, "position_at_finish"]], on=RACE_PROGRAM_ID)

According to the plot below, it seems that zone change happens mostly during the beginning of the race. 

For Stakes race, there's not a significant change in zone after the beginning of the race.

In [ ]:
POSITIONS_TO_COMPARE = 4
FACET_ROW = "race_type_full"
FACET_COL = "position_at_finish"
POSITION_MASK = zone_change.position_at_finish.le(POSITIONS_TO_COMPARE) & zone_change.trakus_segment.gt(3)
GROUP_COLS = [FACET_ROW, FACET_COL, "trakus_segment"]

subset_data = zone_change.loc[POSITION_MASK].groupby(GROUP_COLS)["is_change_zone"].mean().reset_index()

grid = sns.FacetGrid(
    data=subset_data, 
    row=FACET_ROW,
    col=FACET_COL, 
    height=2.5, 
    aspect=1.5,
    margin_titles=True,
    row_order=RACE_TYPES_FULL
)

grid.map(sns.lineplot, "trakus_segment", "is_change_zone")
grid.set_titles(row_template='{row_name}', col_template="Position at finish: {col_name}")

In [ ]:
zone_changes = [
    f"{zone1}_{zone2}"
    for zone1 in tracking_sec.zone.unique()
    for zone2 in tracking_sec.zone.unique()
    if zone1 != zone2
]

In [ ]:
tracking_sec[zone_changes] = pd.DataFrame(
    [
        np.where(
            tracking_sec.zone.eq(change.split("_")[0]) & 
            tracking_sec.groupby(RACE_PROGRAM_ID).zone.shift(1).eq(change.split("_")[1]),
            True,
            False
        )
        for change in zone_changes 
    ]
).transpose().reset_index(drop=True)

In [ ]:
tracking_sec_long = pd.melt(
    tracking_sec, 
    id_vars=[*RACE_PROGRAM_ID, "trakus_sec", "trakus_index", "trakus_segment", "finish_trakus_index", "race_type_full", "dnf"], 
    value_vars=zone_changes, 
    var_name="zone_change",
    value_name="count")

In [ ]:
finish_mask = tracking_sec_long.trakus_index.le(tracking_sec_long.finish_trakus_index) & ~tracking_sec.dnf
zone_change_detailed = tracking_sec_long.groupby([*RACE_PROGRAM_ID,"trakus_segment", "race_type_full", "zone_change"])["count"].sum().reset_index()
zone_change_detailed = zone_change_detailed.merge(start[[*RACE_PROGRAM_ID, "position_at_finish"]], on=RACE_PROGRAM_ID)

In [ ]:
POSITIONS_TO_COMPARE = 1
FACET_COL = "zone_change"
RACE_TYPES = zone_change_detailed.race_type_full.unique()
GROUP_COLS = [FACET_COL, "trakus_segment"]

for race_type in RACE_TYPES:
    print(race_type)
    POSITION_MASK = zone_change_detailed.position_at_finish.le(POSITIONS_TO_COMPARE) & zone_change_detailed.race_type_full.eq(race_type)

    subset_data = zone_change_detailed.loc[POSITION_MASK].groupby(GROUP_COLS)["count"].mean().reset_index()

    subset_data = subset_data[subset_data.groupby("zone_change")["count"].transform("sum").gt(0.05)]

    grid = sns.FacetGrid(
        data=subset_data, 
        col=FACET_COL, 
        col_wrap=5, 
        height=2.5, 
        aspect=1.5,
        margin_titles=True,
    #     row_order=RACE_TYPES_FULL
    )

    grid.map(sns.lineplot, "trakus_segment", "count")
    grid.set_titles(col_template="Zone change: {col_name}")
    plt.show()

## Animated plots (Tar)

## Jockey vs Horse (Why it doesn't work and how to make it work?)

# Suggestions to jockey